In [ ]:
# ============================================================
# WGP Extra Trees: SHAP + ALE, random 80/20 split (seed 42)
# Aesthetic version: same computations, restyled Figures 10-13.
# Install: pip install numpy pandas matplotlib scipy scikit-learn shap
# ============================================================
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_rgb
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.transforms import ScaledTranslation
import shap
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

PROJECT = Path(r"D:\2026 Work\My Papers\waste glass concrete")
OUT = (PROJECT / "Revision of Manuscript" / "1st Revision"
       / "Work" / "SHAP and ALE" / "ExtraTrees_Random42")
OUT.mkdir(parents=True, exist_ok=True)

FEATURES = ["GGP size", "Replacement", "Cement", "Max size", "CA", "FA",
            "WC Ratio", "SiO2", "CaO", "Na2O", "Days"]
TARGET = "CS"

# Display names and units follow Table 2 of Poudel et al. (2026)
DISPLAY = {"GGP size": "WGP size", "Replacement": "Replacement level",
           "Cement": "Cement content", "Max size": "Max. aggregate size",
           "CA": "Coarse aggregate", "FA": "Fine aggregate", "WC Ratio": "W/C ratio",
           "SiO2": "SiO$_2$", "CaO": "CaO", "Na2O": "Na$_2$O", "Days": "Curing time"}
UNITS = {"GGP size": "μm", "Replacement": "%", "Cement": "kg/m³", "Max size": "mm",
         "CA": "kg/m³", "FA": "kg/m³", "WC Ratio": "–", "SiO2": "%", "CaO": "%",
         "Na2O": "%", "Days": "days"}


def lab(f):
    """Axis label with unit, e.g. 'Cement content (kg/m³)'."""
    return f"{DISPLAY[f]} ({UNITS[f]})"


def val_text(f, x):
    """Feature value with unit, e.g. '45 μm', '15%', '0.4'."""
    u = UNITS[f]
    return f"{x:.4g}" if u == "–" else (f"{x:.4g}%" if u == "%" else f"{x:.4g} {u}")
SEED = 42
DPI = 400
ALE_BINS = 10
LOW_SUPPORT = 5

# ---------------- STYLE ----------------
BLUE, BLUE_DARK = "#2D8CFF", "#1B5FB5"
PINK, PINK_DARK = "#F15A7A", "#A63A52"
GRID = "#B5B5B5"
CMAP_FEATURE = LinearSegmentedColormap.from_list(
    "low_high", ["#2D8CFF", "#9B5DE5", "#F15A7A"])
CMAP_SHAP = LinearSegmentedColormap.from_list(
    "shap_div", ["#2D8CFF", "#EEF1F6", "#F15A7A"])

plt.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "stix",
    "font.serif": ["Times New Roman", "Times", "Liberation Serif", "DejaVu Serif"],
    "font.size": 14, "axes.labelsize": 16, "axes.titlesize": 17,
    "xtick.labelsize": 13, "ytick.labelsize": 13, "legend.fontsize": 13,
    "axes.linewidth": 2.0,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.major.size": 5, "ytick.major.size": 5,
    "xtick.major.width": 1.4, "ytick.major.width": 1.4,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def darken(color, factor=0.6):
    return tuple(c * factor for c in to_rgb(color))


def style_axes(ax, grid_axis="both"):
    ax.grid(True, axis=grid_axis, linestyle="--", linewidth=0.8,
            color=GRID, alpha=0.55)
    ax.set_axisbelow(True)


def glossy_scatter(ax, x, y, size=70, color=None, c=None, cmap=None,
                   vmin=None, vmax=None, edge=None, z=3, label=None):
    """Bead-style markers: coloured fill, dark rim, white shine."""
    kw = dict(s=size, linewidths=1.0, alpha=0.95, zorder=z, label=label)
    if c is not None:
        ax.scatter(x, y, c=c, cmap=cmap, vmin=vmin, vmax=vmax,
                   edgecolors=edge or "#3A3A3A", **kw)
    else:
        ax.scatter(x, y, color=color, edgecolors=edge or darken(color), **kw)
    rad = np.sqrt(size / np.pi)
    for scale, shift, alpha in [(0.24, 0.32, 0.5), (0.07, 0.38, 0.9)]:
        off = ScaledTranslation(-shift * rad / 72, shift * rad / 72,
                                ax.figure.dpi_scale_trans)
        ax.scatter(x, y, s=size * scale, color="white", alpha=alpha,
                   linewidths=0, zorder=z + 0.1, transform=ax.transData + off)


def locate_dataset():
    """Find the CSV in the project; prompt if multiple; file dialog if none."""
    preferred = [
        PROJECT / "Data and Main Paper" / "Preprocessed_Data"
        / "01_WGP_database_with_ids(3).csv",
        PROJECT / "Data and Main Paper" / "Preprocessed_Data"
        / "01_WGP_database_with_ids.csv",
        PROJECT / "Data and Main Paper" / "Preprocessed_Data"
        / "Final_data_sushant_preprocessed.csv",
    ]
    matches = [f for f in preferred if f.is_file()]
    matches += [f for f in PROJECT.rglob("01_WGP_database_with_ids*.csv")
                if f.is_file() and f not in matches]
    if not matches:
        print("No matching dataset found under", PROJECT)
        print("Opening file picker; select the CSV on your computer.")
        try:
            from tkinter import Tk, filedialog
            root = Tk()
            root.withdraw()
            root.attributes("-topmost", True)
            choice = filedialog.askopenfilename(
                title="Select 337-row WGP dataset",
                filetypes=[("CSV files", "*.csv")])
            root.destroy()
        except Exception as exc:
            raise FileNotFoundError(
                "File picker unavailable. Place your CSV under the project "
                "folder or edit PROJECT at the top of this script.") from exc
        if not choice:
            raise FileNotFoundError("No CSV selected.")
        return Path(choice)
    if len(matches) == 1:
        return matches[0]
    print("Multiple datasets found:")
    for i, f in enumerate(matches, 1):
        print(f" {i}: {f}")
    while True:
        answer = input("Enter the number of the correct CSV: ").strip()
        if answer.isdigit() and 1 <= int(answer) <= len(matches):
            return matches[int(answer) - 1]
        print("Please enter a listed number.")


def save(fig, filename):
    fig.savefig(OUT / f"{filename}.png", dpi=DPI, bbox_inches="tight",
                facecolor="white")
    fig.savefig(OUT / f"{filename}.pdf", bbox_inches="tight",
                facecolor="white")
    plt.close(fig)


def ale_1d(model, data, feature, bins=ALE_BINS):
    """First-order ALE with quantile bins on original held-out test features.
    Bin support = count of test observations in the interval.
    """
    x = data[feature].to_numpy(dtype=float)
    good = np.isfinite(x)
    data = data.loc[good].reset_index(drop=True)
    x = data[feature].to_numpy(float)
    if len(x) < 3:
        return pd.DataFrame()
    edges = np.unique(np.quantile(x, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return pd.DataFrame()
    which = np.clip(np.searchsorted(edges, x, side="right") - 1,
                    0, len(edges) - 2)
    support = np.bincount(which, minlength=len(edges) - 1)
    differences = np.zeros(len(edges) - 1)
    for b in range(len(differences)):
        ix = np.flatnonzero(which == b)
        if not len(ix):
            continue
        lower = data.iloc[ix].copy()
        upper = lower.copy()
        lower[feature] = edges[b]
        upper[feature] = edges[b + 1]
        differences[b] = np.mean(model.predict(upper) - model.predict(lower))
    cumulative = np.r_[0, np.cumsum(differences)]
    middle = (cumulative[:-1] + cumulative[1:]) / 2
    centered = middle - np.average(middle, weights=support)
    return pd.DataFrame({
        "Feature": feature, "Feature_Value": (edges[:-1] + edges[1:]) / 2,
        "ALE_MPa": centered, "Bin_Support": support,
        "Low_Support": support < LOW_SUPPORT
    })


# ======================= PLOTTING FUNCTIONS =======================
def plot_importance(ranking):
    d = ranking.iloc[::-1]
    vals = d["Mean_Absolute_SHAP_MPa"].to_numpy()
    colors = plt.cm.Blues(0.40 + 0.55 * vals / vals.max())
    fig, ax = plt.subplots(figsize=(9.5, 6.5))
    bars = ax.barh(d["Feature"].map(lab), vals, color=colors, edgecolor="#1B3A66",
                   linewidth=1.2, height=0.68, zorder=3)
    for b, v in zip(bars, vals):
        ax.text(v + vals.max() * 0.015, b.get_y() + b.get_height() / 2,
                f"{v:.2f}", va="center", fontsize=12, fontweight="bold")
    ax.set_xlim(0, vals.max() * 1.15)
    ax.set(xlabel="Mean |SHAP value| (MPa)")
    ax.set_title("Mean feature contribution to predicted compressive strength (testing set)",
                 fontweight="bold", pad=12)
    style_axes(ax, "x")
    save(fig, "Fig_10_SHAP_Importance")


def plot_beeswarm(values, X_test, ranking):
    order = [FEATURES.index(f) for f in ranking["Feature"]]  # most important first
    rng = np.random.default_rng(0)
    fig, ax = plt.subplots(figsize=(11, 8))
    n = len(order)
    for row, j in enumerate(order[::-1]):          # bottom row = least important
        v = values[:, j]
        f = X_test.iloc[:, j].to_numpy(float)
        lo, hi = np.nanpercentile(f, [5, 95])
        norm = np.full_like(f, 0.5) if hi == lo else np.clip((f - lo) / (hi - lo), 0, 1)
        if v.max() > v.min():
            edges = np.linspace(v.min(), v.max(), 31)
            idx = np.digitize(v, edges)
            counts = np.bincount(idx)
            dens = counts[idx] / counts.max()
        else:
            dens = np.ones_like(v)
        y = row + rng.uniform(-1, 1, len(v)) * 0.40 * dens
        ax.axhline(row, color="#E4E4E4", lw=0.8, zorder=0)
        glossy_scatter(ax, v, y, size=62, c=norm, cmap=CMAP_FEATURE,
                       vmin=0, vmax=1, z=3)
    ax.axvline(0, color="#555555", lw=1.4, zorder=1)
    ax.set_yticks(range(n))
    ax.set_yticklabels([lab(FEATURES[j]) for j in order[::-1]])
    ax.set_ylim(-0.7, n - 0.3)
    ax.set(xlabel="SHAP value (impact on compressive strength, MPa)")
    ax.set_title("SHAP value distribution by feature (testing set)",
                 fontweight="bold", pad=12)
    style_axes(ax, "x")
    sm = ScalarMappable(norm=Normalize(0, 1), cmap=CMAP_FEATURE)
    cb = fig.colorbar(sm, ax=ax, pad=0.02, aspect=30)
    cb.set_ticks([0, 1])
    cb.set_ticklabels(["Low", "High"])
    cb.set_label("Feature value", rotation=270, labelpad=18)
    cb.outline.set_linewidth(1.4)
    save(fig, "Fig_10_SHAP_Beeswarm")


def plot_waterfall(values_row, x_row, base_value):
    order = np.argsort(np.abs(values_row))          # bottom (smallest) -> top
    n = len(order)
    fig, ax = plt.subplots(figsize=(11, 7.5))
    cum = base_value
    marks = [cum]
    for k, j in enumerate(order):
        v = values_row[j]
        start, cum = cum, cum + v
        marks.append(cum)
        face, edge = (PINK, PINK_DARK) if v >= 0 else (BLUE, BLUE_DARK)
        ax.barh(k, v, left=start, height=0.62, color=face, edgecolor=edge,
                linewidth=1.3, zorder=3)
        if k < n - 1:
            ax.plot([cum, cum], [k + 0.31, k + 1 - 0.31], color="#777777",
                    lw=0.9, ls=":", zorder=2)
    span = max(marks) - min(marks)
    pad = 0.10 * span + 0.3
    for k, j in enumerate(order):
        v = values_row[j]
        end_x = marks[k + 1] if v >= 0 else marks[k + 1]
        right = max(marks[k], marks[k + 1])
        ax.text(right + pad * 0.15, k, f"{v:+.2f}", va="center", ha="left",
                fontsize=12, fontweight="bold",
                color=PINK_DARK if v >= 0 else BLUE_DARK)
    ax.axvline(base_value, color="#666666", ls="--", lw=1.3, zorder=1)
    ax.axvline(cum, color="#222222", ls="--", lw=1.3, zorder=1)
    ax.text(base_value, -0.95, f"E[f(X)] = {base_value:.2f} MPa", ha="center",
            va="top", fontsize=13, color="#444444", zorder=6,
            bbox=dict(fc="white", ec="none", pad=2.5))
    ax.text(cum, n - 0.25, f"f(x) = {cum:.2f} MPa", ha="center", va="bottom",
            fontsize=13, fontweight="bold", zorder=6,
            bbox=dict(fc="white", ec="none", pad=2.5))
    ax.set_yticks(range(n))
    ax.set_yticklabels([f"{DISPLAY[FEATURES[j]]} = {val_text(FEATURES[j], x_row[j])}" for j in order])
    ax.set_ylim(-1.5, n + 0.5)
    ax.set_xlim(min(marks) - pad, max(marks) + pad * 1.6)
    ax.set(xlabel="Predicted compressive strength (MPa)")
    ax.set_title("Feature contributions for the record with the largest total effect",
                 fontweight="bold", pad=14, fontsize=15)
    style_axes(ax, "x")
    ax.legend(handles=[Patch(facecolor=PINK, edgecolor=PINK_DARK, label="Increases strength"),
                       Patch(facecolor=BLUE, edgecolor=BLUE_DARK, label="Decreases strength")],
              loc="lower right", frameon=True, fancybox=True, edgecolor="#BBBBBB")
    save(fig, "Fig_11_SHAP_Waterfall")


def binned_trend(x, y, bins=8):
    edges = np.unique(np.quantile(x, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return None, None
    idx = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, len(edges) - 2)
    cx, cy = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.any():
            cx.append(x[m].mean())
            cy.append(y[m].mean())
    return np.array(cx), np.array(cy)


def plot_dependence(values, X_test):
    fig, axes = plt.subplots(4, 3, figsize=(16, 18), constrained_layout=True)
    for j, f in enumerate(FEATURES):
        ax = axes.flat[j]
        x = X_test[f].to_numpy(float)
        v = values[:, j]
        pos = v >= 0
        glossy_scatter(ax, x[pos], v[pos], size=75, color=PINK, z=3)
        glossy_scatter(ax, x[~pos], v[~pos], size=75, color=BLUE, z=3)
        cx, cy = binned_trend(x, v)
        if cx is not None:
            ax.plot(cx, cy, color="#222222", lw=2.0, zorder=4)
            ax.plot(cx, cy, "o", color="white", mec="#222222", mew=1.5,
                    ms=6, zorder=5)
        ax.axhline(0, color="#666666", lw=1.2, zorder=1)
        ax.set(xlabel=lab(f), ylabel="SHAP value (MPa)")
        ax.set_title(DISPLAY[f], fontweight="bold")
        style_axes(ax)
    last = axes.flat[len(FEATURES)]
    last.axis("off")
    last.legend(handles=[
        Line2D([], [], ls="", marker="o", mfc=PINK, mec=PINK_DARK, ms=12,
               label="Increases strength (SHAP > 0)"),
        Line2D([], [], ls="", marker="o", mfc=BLUE, mec=BLUE_DARK, ms=12,
               label="Decreases strength (SHAP < 0)"),
        Line2D([], [], color="#222222", lw=2, marker="o", mfc="white",
               mec="#222222", label="Binned mean trend")],
        loc="center", frameon=True, fancybox=True, edgecolor="#BBBBBB", fontsize=14)
    fig.suptitle("Effect of each input on predicted compressive strength (SHAP, testing set)",
                 fontsize=20, fontweight="bold")
    save(fig, "Fig_12_SHAP_Dependence")


def plot_ale(ale, X_test):
    fig, axes = plt.subplots(4, 3, figsize=(17, 18), constrained_layout=True)
    for j, f in enumerate(FEATURES):
        ax = axes.flat[j]
        d = ale[ale["Feature"] == f]
        x, y = d["Feature_Value"].to_numpy(), d["ALE_MPa"].to_numpy()
        ax.fill_between(x, 0, y, color=BLUE, alpha=0.14, zorder=1)
        ax.plot(x, y, color=BLUE_DARK, lw=2.4, zorder=2)
        ok = ~d["Low_Support"].to_numpy()
        glossy_scatter(ax, x[ok], y[ok], size=95, color=BLUE, z=3)
        if (~ok).any():
            ax.scatter(x[~ok], y[~ok], marker="X", s=130, color=PINK,
                       edgecolors=PINK_DARK, linewidths=1.2, zorder=5)
        ax.axhline(0, color="#666666", lw=1.2, zorder=1)
        # rug of held-out observations along the bottom edge
        ax.plot(X_test[f].to_numpy(float), np.full(len(X_test), 0.012), "|",
                color="#555555", alpha=0.5, ms=9, mew=1.0,
                transform=ax.get_xaxis_transform(), clip_on=False, zorder=2)
        ax.set(xlabel=lab(f), ylabel="ALE (MPa)")
        ax.set_title(DISPLAY[f], fontweight="bold")
        style_axes(ax)
    last = axes.flat[len(FEATURES)]
    last.axis("off")
    last.legend(handles=[
        Line2D([], [], color=BLUE_DARK, lw=2.4, marker="o", mfc=BLUE,
               mec=BLUE_DARK, ms=11, label="ALE curve (bin centres)"),
        Line2D([], [], ls="", marker="X", mfc=PINK, mec=PINK_DARK, ms=12,
               label=f"Low-support bin (n < {LOW_SUPPORT})"),
        Line2D([], [], ls="", marker="|", color="#555555", ms=12,
               label="Held-out observations (rug)")],
        loc="center", frameon=True, fancybox=True, edgecolor="#BBBBBB", fontsize=14)
    fig.suptitle("Accumulated local effects on compressive strength (testing set)",
                 fontsize=20, fontweight="bold")
    save(fig, "Fig_13_ALE_Curves")


# ============================== MAIN ==============================
def main():
    dataset = locate_dataset()
    print("Using dataset:", dataset)
    df = pd.read_csv(dataset)
    missing = set(FEATURES + [TARGET]) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if len(df) != 337:
        raise ValueError(f"Expected 337 records, found {len(df)}.")
    X = df[FEATURES].apply(pd.to_numeric, errors="coerce")
    y = pd.to_numeric(df[TARGET], errors="coerce")
    if y.isna().any():
        raise ValueError("Target CS contains missing/non-numeric values.")

    ids = np.arange(len(df))
    train_idx, test_idx = train_test_split(ids, test_size=.2, random_state=SEED)
    X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    pd.DataFrame({"original_index": train_idx}).to_csv(
        OUT / "Training_Indices.csv", index=False)
    pd.DataFrame({"original_index": test_idx}).to_csv(
        OUT / "Testing_Indices.csv", index=False)

    imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    X_train = pd.DataFrame(imputer.fit_transform(X_train_raw),
                           columns=FEATURES, index=X_train_raw.index)
    X_test = pd.DataFrame(imputer.transform(X_test_raw),
                          columns=FEATURES, index=X_test_raw.index)
    model = ExtraTreesRegressor(
        n_estimators=200, min_samples_split=10, min_samples_leaf=2,
        max_features=1.0, max_depth=20, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    print("Train n =", len(X_train), "; test n =", len(X_test))
    print("Test R2 =", round(r2_score(y_test, predictions), 4))
    print("Test RMSE =", round(np.sqrt(mean_squared_error(y_test, predictions)), 4))
    print("Test MAE =", round(mean_absolute_error(y_test, predictions), 4))
    pd.DataFrame({"original_index": test_idx, "Actual_CS": y_test.to_numpy(),
                  "Predicted_CS": predictions}).to_csv(
                      OUT / "ExtraTrees_Test_Predictions.csv", index=False)

    # Tree SHAP is appropriate for ExtraTreesRegressor; additive in MPa.
    explainer = shap.TreeExplainer(model)
    explanation = explainer(X_test)
    values = np.asarray(explanation.values)
    base = np.asarray(explanation.base_values).reshape(-1)
    if base.size == 1:
        base = np.repeat(base, len(X_test))
    if values.shape != (len(X_test), len(FEATURES)):
        raise ValueError(f"Unexpected SHAP shape: {values.shape}")
    max_gap = np.max(np.abs(base + values.sum(axis=1) - predictions))
    print("SHAP additivity max absolute difference (MPa):", max_gap)
    if max_gap > 1e-4:
        warnings.warn("Unexpected SHAP additivity mismatch")

    table = X_test.reset_index(drop=True).copy()
    table.insert(0, "original_index", test_idx)
    table["Actual_CS"] = y_test.to_numpy()
    table["Predicted_CS"] = predictions
    table["Base_Value"] = base
    for j, f in enumerate(FEATURES):
        table["SHAP_" + f] = values[:, j]
    table.to_csv(OUT / "SHAP_Values_Testing.csv", index=False)
    ranking = pd.DataFrame({"Feature": FEATURES,
                            "Mean_Absolute_SHAP_MPa": np.abs(values).mean(axis=0)})
    ranking.sort_values("Mean_Absolute_SHAP_MPa", ascending=False, inplace=True)
    ranking.to_csv(OUT / "SHAP_Feature_Importance.csv", index=False)

    # ---- Figures 10-11 ----
    plot_importance(ranking)
    plot_beeswarm(values, X_test, ranking)

    example = int(np.argmax(np.abs(values).sum(axis=1)))
    plot_waterfall(values[example], X_test.iloc[example].to_numpy(float),
                   float(base[example]))
    pd.DataFrame([{
        "original_index": int(test_idx[example]),
        "Actual_CS": float(y_test.iloc[example]),
        "Predicted_CS": float(predictions[example]),
        "Base_Value": float(base[example]),
        "SHAP_Sum": float(values[example].sum()),
        "FA": float(X_test.iloc[example]["FA"]),
        "WC_Ratio": float(X_test.iloc[example]["WC Ratio"])
    }]).to_csv(OUT / "Waterfall_Example_Audit.csv", index=False)
    pd.DataFrame({
        "Feature": ["FA", "WC Ratio"],
        "Dataset_Minimum": [X["FA"].min(), X["WC Ratio"].min()],
        "Reviewer_Claim": [371.7, .35],
        "Matches_Claim": np.isclose([X["FA"].min(), X["WC Ratio"].min()],
                                    [371.7, .35])
    }).to_csv(OUT / "Dataset_Minimum_Audit.csv", index=False)

    # ---- Figure 12: dependence ----
    plot_dependence(values, X_test)

    # ---- Figure 13: ALE ----
    ale_tables = [ale_1d(model, X_test, f) for f in FEATURES]
    ale_tables = [d for d in ale_tables if not d.empty]
    ale = pd.concat(ale_tables, ignore_index=True)
    ale.to_csv(OUT / "ALE_Numerical_Results.csv", index=False)
    plot_ale(ale, X_test)

    # G6: fixed-model sensitivity, not retraining.
    scenarios = {
        "All": np.ones(len(X_test), dtype=bool),
        "Without_300_Days": ~np.isclose(X_test["Days"].to_numpy(), 300),
        "Without_Minimum_FA": ~np.isclose(X_test["FA"].to_numpy(), X["FA"].min()),
        "Without_Both": (~np.isclose(X_test["Days"].to_numpy(), 300) &
                         ~np.isclose(X_test["FA"].to_numpy(), X["FA"].min()))
    }
    sens = []
    for name, mask in scenarios.items():
        if not mask.any():
            continue
        for j, f in enumerate(FEATURES):
            sens.append({"Scenario": name, "Feature": f, "N_Test": int(mask.sum()),
                         "Mean_Absolute_SHAP_MPa": np.abs(values[mask, j]).mean()})
        parts = [ale_1d(model, X_test.iloc[np.flatnonzero(mask)], f)
                 for f in FEATURES]
        parts = [d for d in parts if not d.empty]
        if parts:
            pd.concat(parts, ignore_index=True).to_csv(
                OUT / f"ALE_{name}.csv", index=False)
    pd.DataFrame(sens).to_csv(OUT / "SHAP_Sensitivity_Analysis.csv", index=False)

    audit = []
    for f in ["Replacement", "FA"]:
        j = FEATURES.index(f)
        rho, p = spearmanr(X_test[f], values[:, j])
        d = ale[ale["Feature"] == f]
        audit.append({"Feature": f, "SHAP_Spearman_Rho": rho, "p_value": p,
                      "ALE_Min": d["ALE_MPa"].min(),
                      "ALE_Max": d["ALE_MPa"].max(),
                      "Low_Support_Bins": int(d["Low_Support"].sum())})
    pd.DataFrame(audit).to_csv(OUT / "Replacement_FA_Audit.csv", index=False)

    (OUT / "METHODOLOGY_NOTES.txt").write_text(
        "Random 80:20 split, seed 42; Extra Trees 200 estimators, "
        "min_samples_split=10, min_samples_leaf=2, max_features=1.0, "
        "max_depth=20. TreeSHAP and ALE use 68 test observations; "
        "ALE bin support means number of held-out observations in a bin. "
        "SHAP base value is E[f(X)] under TreeExplainer's default "
        "tree-path-dependent background, NOT the mean of a specified "
        "external background dataset. The waterfall record is selected "
        "by largest sum of absolute SHAP values. "
        "Sensitivity removes records from the explanation set, "
        "not from model fitting. No 18-fold stability analysis or "
        "exact SHAP interaction analysis is included. "
        "Check the selected dataset and confirm the random split "
        "matches the manuscript's original split.",
        encoding="utf-8")
    print("Completed. Files saved to:", OUT)


if __name__ == "__main__":
    main()

Multiple datasets found:
 1: D:\2026 Work\My Papers\waste glass concrete\Data and Main Paper\Preprocessed_Data\Final_data_sushant_preprocessed.csv
 2: D:\2026 Work\My Papers\waste glass concrete\Revision of Manuscript\1st Revision\New folder\01_WGP_database_with_ids.csv
 3: D:\2026 Work\My Papers\waste glass concrete\Revision of Manuscript\1st Revision\Work\Data\01_WGP_database_with_ids.csv


Enter the number of the correct CSV:  3


Using dataset: D:\2026 Work\My Papers\waste glass concrete\Revision of Manuscript\1st Revision\Work\Data\01_WGP_database_with_ids.csv
Train n = 269 ; test n = 68
Test R2 = 0.8224
Test RMSE = 5.2426
Test MAE = 3.8075
SHAP additivity max absolute difference (MPa): 1.1368683772161603e-13
